# Try TimesFM

Google Research's time-series foundation model: https://github.com/google-research/timesfm

Tip: **Runtime → Change runtime type → T4 GPU** makes it faster (CPU works too).

Run the cells top to bottom (Shift+Enter).

In [ ]:
!pip install -q "timesfm[torch] @ git+https://github.com/google-research/timesfm.git"

## Load the model
The first run downloads the weights (~1 GB) from Hugging Face.

TimesFM 3.0 weights are for non-commercial use only. For commercial use, switch to the Apache-2.0 TimesFM 2.5 model (`timesfm.TimesFM_2p5_200M_torch`, `google/timesfm-2.5-200m-pytorch`).

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from timesfm import TimesFM3Forecaster

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
forecaster = TimesFM3Forecaster.from_pretrained("google/timesfm-3.0-pytorch", device=device)

## Your data

By default this uses a synthetic series (trend + weekly seasonality + noise).

To use your own data, set `USE_CSV = True` and run the cell. It will ask you to upload a CSV, then set `VALUE_COLUMN` to the name of the column you want to forecast. Rows must be in time order.

In [ ]:
USE_CSV = False
VALUE_COLUMN = "value"
HORIZON = 30  # how many steps ahead to forecast

if USE_CSV:
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
    series = df[VALUE_COLUMN].astype(np.float32).to_numpy()
else:
    t = np.arange(365)
    rng = np.random.default_rng(0)
    series = (100 + 0.2 * t + 10 * np.sin(2 * np.pi * t / 7) + rng.normal(0, 2, t.size)).astype(np.float32)

print(len(series), "points")

## Forecast

In [ ]:
out = forecaster.predict(series, horizon=HORIZON, return_quantiles=True)
median, q = out.forecast, out.quantiles  # q[:, 0] = 10th pct, q[:, 8] = 90th pct

x_hist = np.arange(len(series))
x_fut = np.arange(len(series), len(series) + HORIZON)
plt.figure(figsize=(12, 4))
plt.plot(x_hist, series, label="history")
plt.plot(x_fut, median, label="forecast (median)")
plt.fill_between(x_fut, q[:, 0], q[:, 8], alpha=0.3, label="10–90% interval")
plt.legend(); plt.show()

pd.DataFrame({"step": np.arange(1, HORIZON + 1), "median": median, "p10": q[:, 0], "p90": q[:, 8]}).round(2)